# DiffSinger Miadu Fine-tuning (Python 3.12 終極相容版)
**本版本特點**：
- **嚴格同步**：會檢查數據完整性，確保不會跳過拷貝。
- **徹底解決 `webrtcvad` 崩潰問題**：代碼層級優雅降級。

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：獲取代碼與環境配置

In [ ]:
%cd /content/
!git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
%cd diffsinger-repo
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0

## 第三步：建立安全連接 (嚴格檢查版)

In [ ]:
import os
drive_root = "/content/drive/MyDrive/DiffSinger_Colab"
if not os.path.exists(drive_root):
    drive_root = "/content/drive/MyDrive/DiffSinger _Colab"
print(f'| 使用 Drive 路徑: {drive_root}')

!mkdir -p "{drive_root}/checkpoints_finetune"
!rm -rf checkpoints
!ln -s "{drive_root}/checkpoints_finetune" checkpoints
!mkdir -p data/binary

print('| 正在檢查數據完整性...')

# 1. 檢查基礎權重完整性
if not os.path.exists("checkpoints/1117_opencpop_ds1000_strict_pinyin/model_ckpt_steps_200000.ckpt"):
    print("| 正在同步權重...")
    !cp -rv "{drive_root}/1117_opencpop_ds1000_strict_pinyin" checkpoints/
    !cp -rv "{drive_root}/nsf_hifigan_44.1k_hop512_128bin_2024.02" checkpoints/

# 2. 檢查數據完整性 (檢查核心檔案)
if not os.path.exists("data/binary/miadu_finetune/train.data"):
    print("| 正在同步二進位數據 (這可能需要幾分鐘)...")
    !rm -rf data/binary/miadu_finetune
    !cp -rv "{drive_root}/miadu_finetune" data/binary/

if os.path.exists("data/binary/miadu_finetune/phone_set.json"):
    size = os.path.getsize("data/binary/miadu_finetune/phone_set.json")
    if size == 0:
        print("| 錯誤：發現 0 字節的 phone_set.json，正在嘗試重新同步...")
        !cp -v "{drive_root}/miadu_finetune/phone_set.json" data/binary/miadu_finetune/

print("| 環境就緒！")

## 第四步：啟動訓練

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1